In [1]:
# =========================================================
# BƯỚC 1: CÀI ĐẶT & ÉP PYTHON NHẬN DIỆN THƯ MỤC
# =========================================================
import os
import sys
import json
import csv
from types import ModuleType

import torch
from torchvision.ops import nms

%cd /kaggle/working

if not os.path.exists("DocLayout-YOLO"):
    !git clone --depth 1 https://github.com/opendatalab/DocLayout-YOLO.git

%cd DocLayout-YOLO
!pip install -q -e .

repo_dir = "/kaggle/working/DocLayout-YOLO"
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

%cd /kaggle/working

# =========================================================
# BƯỚC 2: "HACK" BỘ NHỚ ĐỂ FIX LỖI MODULE HUB
# =========================================================
if "doclayout_yolo.utils.callbacks.hub" not in sys.modules:
    dummy_hub = ModuleType("doclayout_yolo.utils.callbacks.hub")
    dummy_hub.callbacks = {}
    sys.modules["doclayout_yolo.utils.callbacks.hub"] = dummy_hub

/kaggle/working
Cloning into 'DocLayout-YOLO'...
remote: Enumerating objects: 277, done.
remote: Counting objects: 100% (277/277), done.
remote: Compressing objects: 100% (234/234), done.
remote: Total 277 (delta 41), reused 237 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (277/277), 10.79 MiB | 25.57 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/kaggle/working/DocLayout-YOLO
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 79.9 MB/s eta 0:00:00
  Building editable for doclayout_yolo (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 w

In [2]:
from pathlib import Path
import shutil
import torch
from torchvision.ops import nms
from doclayout_yolo import YOLOv10

print("✅ Import YOLOv10 thành công!")

# ---------------------------------------------------------
# HÀM HẬU XỬ LÝ BOX: NMS + PADDING (ĐÃ THÊM KEY 'score')
# ---------------------------------------------------------
def postprocess_boxes(results_list,
                      img_w: int,
                      img_h: int,
                      names,
                      iou_thr: float = 0.6,
                      pad_scale_x: float = 0.0,
                      pad_scale_y: float = 0.0,
                      agnostic_nms: bool = True):
    boxes = []
    scores = []
    labels = []

    for r in results_list:
        for box in r.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            conf = float(box.conf[0])
            cls_id = int(box.cls[0])

            boxes.append([x1, y1, x2, y2])
            scores.append(conf)
            labels.append(cls_id)

    if len(boxes) == 0:
        return []

    boxes = torch.tensor(boxes)
    scores = torch.tensor(scores)
    labels = torch.tensor(labels)

    keep_indices = []

    if agnostic_nms:
        kept = nms(boxes, scores, iou_thr)
        keep_indices = kept.tolist()
    else:
        unique_classes = set(labels.tolist())
        for cls in unique_classes:
            cls_mask = labels == cls
            cls_indices = torch.nonzero(cls_mask, as_tuple=True)[0]
            cls_boxes = boxes[cls_indices]
            cls_scores = scores[cls_indices]

            kept = nms(cls_boxes, cls_scores, iou_thr)
            keep_indices.extend(cls_indices[kept].tolist())

        keep_indices = sorted(set(keep_indices))

    regions = []
    for idx in keep_indices:
        x1, y1, x2, y2 = boxes[idx].tolist()
        h = y2 - y1

        pad_x = pad_scale_x * h
        pad_y = pad_scale_y * h

        x1 = max(0.0, x1 - pad_x)
        y1 = max(0.0, y1 - pad_y)
        x2 = min(float(img_w), x2 + pad_x)
        y2 = min(float(img_h), y2 + pad_y)

        cls_id = int(labels[idx])
        cls_name = names[cls_id] if names is not None else "region"
        conf_score = float(scores[idx])

        regions.append({
            "bbox": [round(x1, 2), round(y1, 2), round(x2, 2), round(y2, 2)],
            "type": cls_name,
            "score": conf_score,  # Keep full precision for AP ranking; round only in reports if needed.
            "text": ""
        })

    return regions

✅ Import YOLOv10 thành công!


In [3]:
!pip install -q scipy pycocotools

import json
import csv
import shutil
import platform
import torch
import torchvision
import numpy as np
from collections import defaultdict, Counter
from contextlib import redirect_stdout
from importlib.metadata import PackageNotFoundError, version
from io import StringIO
from pathlib import Path

from scipy.optimize import linear_sum_assignment
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# [SỬA ĐIỂM 1]: Thêm PIL và bật chế độ bỏ qua lỗi metadata/header của JPEG
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True 

def get_package_version(package_name):
    try:
        return version(package_name)
    except PackageNotFoundError:
        return "not-installed"

# ---------------------------------------------------------
# CONFIGURATION: PATHS AND PARAMETERS
# ---------------------------------------------------------
MODEL_PATH_1 = "/kaggle/input/datasets/notpitomon/htd-final-bbox-weight/Doclayout Yolo Final V 2.2.pt"
MODEL_PATH_2 = "/kaggle/input/datasets/notpitomon/htd-final-bbox-weight/Doclayout Yolo Final V 2.1.pt"
TEST_IMG_DIR = Path("/kaggle/input/datasets/quii29/rukopys-dataset/train/images")
GT_META_PATH = Path("/kaggle/input/datasets/notpitomon/htd-final-better-gold-dataset/test.jsonl")

# [SỬA ĐIỂM 2]: Đọc GT_META_PATH ngay tại đây để tạo list ảnh chuẩn
print("Reading ground-truth metadata...")
gt_records = []
gt_data = {}
with open(GT_META_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        gt_records.append(item)
        gt_data[Path(item['file_name']).name] = item['regions']

image_paths = [TEST_IMG_DIR / Path(item['file_name']).name for item in gt_records]
missing_on_disk = [p.name for p in image_paths if not p.exists()]
if missing_on_disk:
    raise FileNotFoundError(f"Có {len(missing_on_disk)} ảnh có trong test.jsonl nhưng không tồn tại trong folder: {missing_on_disk[:5]}...")

OUT_DIR = Path("/kaggle/working/inference_output")
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

SUB_PATH = OUT_DIR / "submission.csv"
SCORED_PRED_M1_PATH = OUT_DIR / "4.1 conf 0.20.csv"
SCORED_PRED_M2_PATH = OUT_DIR / "4.2 conf 0.20.csv"
SCORED_PRED_ENS_PATH = OUT_DIR / "ensemble conf 0.20.csv"
SCORED_PRED_PATH = SCORED_PRED_ENS_PATH  # Backward-compatible alias for the primary ensemble artifact.
REPORT_PATH = OUT_DIR / "detector_paper_report.txt"

IMG_SIZE = 1280
# CONF = 0.001  # Paper-grade AP needs broad confidence ranking; avoid truncating recall early.
CONF = 0.2  # Đây để chạy lấy kq
MAX_DET = 300
IOU_NMS = 0.55
COCO_MAX_DETS = [1, 10, 100]

REPORT_CLASSES = ['handwritten', 'formula', 'printed', 'annotation', 'table', 'image', 'graph']
MODEL_CLASSES = None  # Populated after model load; this is the YOLO class-id order.
CLASS_TO_ID = {cls: i + 1 for i, cls in enumerate(REPORT_CLASSES)}
ID_TO_CLASS = {v: k for k, v in CLASS_TO_ID.items()}

RUN_INFO = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "scipy": get_package_version("scipy"),
    "pycocotools": get_package_version("pycocotools"),
}

print(f"Found exactly {len(image_paths)} GT images for inference.")

# ---------------------------------------------------------
# EVALUATION HELPERS
# ---------------------------------------------------------
def image_key(file_name):
    return Path(file_name).name


def xyxy_to_xywh(box):
    x1, y1, x2, y2 = [float(v) for v in box]
    w = max(0.0, x2 - x1)
    h = max(0.0, y2 - y1)
    return [x1, y1, w, h], w * h


def normalize_model_names(names):
    if isinstance(names, dict):
        return [str(names[k]) for k in sorted(names.keys(), key=lambda x: int(x))]
    return [str(name) for name in list(names)]


def validate_model_class_mapping(names, report_classes, model_name):
    actual_classes = normalize_model_names(names)
    expected_set = set(report_classes)
    actual_counts = Counter(actual_classes)
    duplicate_actual = sorted([name for name, count in actual_counts.items() if count > 1])
    missing_classes = sorted(expected_set - set(actual_classes))
    extra_classes = sorted(set(actual_classes) - expected_set)

    if duplicate_actual or missing_classes or extra_classes:
        raise ValueError(
            f"{model_name}: class set mismatch. "
            f"model_order={actual_classes}, report_classes={list(report_classes)}, "
            f"duplicates={duplicate_actual}, missing={missing_classes}, extra={extra_classes}."
        )
    return actual_classes


def validate_evaluation_image_set(gt_records, image_paths):
    gt_names = [image_key(item['file_name']) for item in gt_records]
    image_names = [p.name for p in image_paths]
    gt_counts = Counter(gt_names)
    image_counts = Counter(image_names)

    duplicate_gt = sorted([name for name, count in gt_counts.items() if count > 1])
    duplicate_images = sorted([name for name, count in image_counts.items() if count > 1])
    if duplicate_gt or duplicate_images:
        raise ValueError(
            "Duplicate image names found in evaluation inputs. "
            f"duplicate_gt={duplicate_gt[:10]}, duplicate_images={duplicate_images[:10]}"
        )

    gt_set = set(gt_names)
    image_set = set(image_names)
    missing_images = sorted(gt_set - image_set)
    extra_images = sorted(image_set - gt_set)
    if missing_images or extra_images:
        raise ValueError(
            "Evaluation image set mismatch between GT metadata and inference directory. "
            f"missing_images={missing_images[:10]} (total={len(missing_images)}), "
            f"extra_images={extra_images[:10]} (total={len(extra_images)})."
        )

    return {
        "num_gt_images": len(gt_names),
        "num_inference_images": len(image_names),
        "exact_match": True,
    }


def assert_prediction_image_set(pred_dict, image_id_by_name, model_name):
    pred_set = set(pred_dict.keys())
    gt_set = set(image_id_by_name.keys())
    missing_predictions = sorted(gt_set - pred_set)
    extra_predictions = sorted(pred_set - gt_set)
    if missing_predictions or extra_predictions:
        raise ValueError(
            f"{model_name}: prediction image set does not match GT image set. "
            f"missing_predictions={missing_predictions[:10]} (total={len(missing_predictions)}), "
            f"extra_predictions={extra_predictions[:10]} (total={len(extra_predictions)})."
        )


def validate_score(raw_score, model_name, img_name, cls_name):
    if raw_score is None:
        return None, "missing"
    try:
        score = float(raw_score)
    except (TypeError, ValueError):
        return None, "non_numeric"
    if not np.isfinite(score):
        return None, "non_finite"
    if score < 0.0 or score > 1.0:
        return None, "out_of_range"
    return score, None


def compute_iou_single(box1, box2):
    x1, y1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    x2, y2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area1 = max(0.0, box1[2] - box1[0]) * max(0.0, box1[3] - box1[1])
    area2 = max(0.0, box2[2] - box2[0]) * max(0.0, box2[3] - box2[1])
    denom = area1 + area2 - inter
    return inter / denom if denom > 0 else 0.0


def compute_confusion_matrix_hungarian(gt_dict, pred_dict, iou_thresh=0.5):
    cm = defaultdict(int)
    for img_name, gts in gt_dict.items():
        preds = pred_dict.get(img_name, [])
        g_valid = [g for g in gts if g['type'] in CLASS_TO_ID]
        p_valid = [p for p in preds if p['type'] in CLASS_TO_ID]
        M, N = len(g_valid), len(p_valid)

        if M == 0 and N == 0:
            continue
        if M == 0:
            for p in p_valid:
                cm[("background", p['type'])] += 1
            continue
        if N == 0:
            for g in g_valid:
                cm[(g['type'], "missed")] += 1
            continue

        iou_mat = np.zeros((N, M), dtype=np.float32)
        for i, p in enumerate(p_valid):
            for j, g in enumerate(g_valid):
                iou_mat[i, j] = compute_iou_single(p['bbox'], g['bbox'])

        row_ind, col_ind = linear_sum_assignment(-iou_mat)
        matched_p, matched_g = set(), set()

        for p_idx, g_idx in zip(row_ind, col_ind):
            if iou_mat[p_idx, g_idx] >= iou_thresh:
                cm[(g_valid[g_idx]['type'], p_valid[p_idx]['type'])] += 1
                matched_p.add(p_idx)
                matched_g.add(g_idx)

        for j in range(M):
            if j not in matched_g:
                cm[(g_valid[j]['type'], "missed")] += 1
        for i in range(N):
            if i not in matched_p:
                cm[("background", p_valid[i]['type'])] += 1
    return cm


def build_coco_ground_truth(gt_records):
    images, annotations = [], []
    image_id_by_name = {}
    gt_counts = Counter()
    ann_id = 1

    for img_id, item in enumerate(gt_records, 1):
        name = image_key(item['file_name'])
        image_id_by_name[name] = img_id
        images.append({
            "id": img_id,
            "file_name": name,
            "width": int(item.get('image_width', 0)),
            "height": int(item.get('image_height', 0)),
        })

        for region in item.get('regions', []):
            cls_name = region.get('type')
            if cls_name not in CLASS_TO_ID:
                continue
            bbox_xywh, area = xyxy_to_xywh(region['bbox'])
            if area <= 0:
                continue
            gt_counts[cls_name] += 1
            annotations.append({
                "id": ann_id,
                "image_id": img_id,
                "category_id": CLASS_TO_ID[cls_name],
                "bbox": bbox_xywh,
                "area": area,
                "iscrowd": 0,
            })
            ann_id += 1

    coco_gt = COCO()
    coco_gt.dataset = {
        "info": {"description": "HTD detector evaluation ground truth"},
        "licenses": [],
        "images": images,
        "annotations": annotations,
        "categories": [
            {"id": CLASS_TO_ID[name], "name": name, "supercategory": "region"}
            for name in REPORT_CLASSES
        ],
    }
    coco_gt.createIndex()
    return coco_gt, image_id_by_name, gt_counts


def build_coco_detections(pred_dict, image_id_by_name, model_name):
    detections = []
    pred_counts = Counter()
    score_issue_counts = Counter()
    score_issue_examples = []

    for img_name, regions in pred_dict.items():
        if img_name not in image_id_by_name:
            continue
        img_id = image_id_by_name[img_name]
        for region in regions:
            cls_name = region.get('type')
            if cls_name not in CLASS_TO_ID:
                continue
            bbox_xywh, area = xyxy_to_xywh(region['bbox'])
            if area <= 0:
                continue

            score, issue = validate_score(region.get('score'), model_name, img_name, cls_name)
            if issue:
                score_issue_counts[issue] += 1
                if len(score_issue_examples) < 5:
                    score_issue_examples.append({"image": img_name, "class": cls_name, "issue": issue})
                continue

            detections.append({
                "image_id": img_id,
                "category_id": CLASS_TO_ID[cls_name],
                "bbox": bbox_xywh,
                "score": score,
            })
            pred_counts[cls_name] += 1

    return detections, pred_counts, score_issue_counts, score_issue_examples


def evaluate_model_cocoeval(pred_dict, gt_records, model_name):
    coco_gt, image_id_by_name, gt_counts = build_coco_ground_truth(gt_records)
    assert_prediction_image_set(pred_dict, image_id_by_name, model_name)
    detections, pred_counts, score_issue_counts, score_issue_examples = build_coco_detections(
        pred_dict, image_id_by_name, model_name
    )

    if score_issue_counts:
        raise ValueError(
            f"{model_name}: invalid scores found before COCOeval. "
            f"counts={dict(score_issue_counts)}, examples={score_issue_examples}. "
            "Official COCO mAP requires real finite confidence scores in [0, 1]."
        )
    if not detections:
        raise ValueError(f"{model_name}: has no valid predictions for COCOeval.")

    coco_dt = coco_gt.loadRes(detections)
    evaluator = COCOeval(coco_gt, coco_dt, 'bbox')
    evaluator.params.imgIds = sorted(image_id_by_name.values())
    evaluator.params.catIds = [CLASS_TO_ID[name] for name in REPORT_CLASSES]
    evaluator.params.maxDets = COCO_MAX_DETS
    evaluator.evaluate()
    evaluator.accumulate()

    summary_buffer = StringIO()
    with redirect_stdout(summary_buffer):
        evaluator.summarize()

    precision = evaluator.eval['precision']  # [IoU, recall, class, area, maxDet]
    per_class = []
    for k, cls_name in enumerate(REPORT_CLASSES):
        p_all = precision[:, :, k, 0, 2]
        p_50 = precision[0, :, k, 0, 2]
        ap = float(np.mean(p_all[p_all > -1])) if np.any(p_all > -1) else float('nan')
        ap50 = float(np.mean(p_50[p_50 > -1])) if np.any(p_50 > -1) else float('nan')
        per_class.append({
            "class": cls_name,
            "ap50": ap50,
            "ap50_95": ap,
            "gt": gt_counts[cls_name],
            "pred": pred_counts[cls_name],
        })

    return {
        "model_name": model_name,
        "map50_95": float(evaluator.stats[0]),
        "map50": float(evaluator.stats[1]),
        "map75": float(evaluator.stats[2]),
        "ar100": float(evaluator.stats[8]),
        "summary": summary_buffer.getvalue().strip(),
        "per_class": per_class,
        "num_gt_boxes": int(sum(gt_counts.values())),
        "num_pred_boxes": int(sum(pred_counts.values())),
        "num_images": len(image_id_by_name),
    }

# ---------------------------------------------------------
# RUN INFERENCE AND VALIDATION
# ---------------------------------------------------------
image_set_summary = validate_evaluation_image_set(gt_records, image_paths)
print(
    "Image set validation passed: "
    f"{image_set_summary['num_gt_images']} GT images == "
    f"{image_set_summary['num_inference_images']} inference images."
)

print("\nLoading models...")
model1 = YOLOv10(MODEL_PATH_1)
model2 = YOLOv10(MODEL_PATH_2)
model1_classes = validate_model_class_mapping(model1.names, REPORT_CLASSES, "V1.0 (V4.1 only)")
model2_classes = validate_model_class_mapping(model2.names, REPORT_CLASSES, "V1.1 (V4.2 only)")
if model1_classes != model2_classes:
    raise ValueError(f"Model class-id order differs: model1={model1_classes}, model2={model2_classes}")
MODEL_CLASSES = model1_classes
print(f"Model class-id order validation passed: {MODEL_CLASSES}")
print(f"Report/COCO category order: {REPORT_CLASSES}")

preds_m1, preds_m2, preds_ens = {}, {}, {}

print("Starting inference...")
with open(SUB_PATH, 'w', newline='', encoding='utf-8') as f_sub, \
     open(SCORED_PRED_M1_PATH, 'w', newline='', encoding='utf-8') as f_m1, \
     open(SCORED_PRED_M2_PATH, 'w', newline='', encoding='utf-8') as f_m2, \
     open(SCORED_PRED_ENS_PATH, 'w', newline='', encoding='utf-8') as f_ens:
    sub_writer = csv.writer(f_sub)
    m1_writer = csv.writer(f_m1)
    m2_writer = csv.writer(f_m2)
    ens_writer = csv.writer(f_ens)

    sub_writer.writerow(["image", "regions"])
    m1_writer.writerow(["image", "regions"])
    m2_writer.writerow(["image", "regions"])
    ens_writer.writerow(["image", "regions"])

    for idx, img_path in enumerate(image_paths, 1):
        # [SỬA ĐIỂM 3]: Dùng PIL đọc ảnh trực tiếp vào RAM để triệt tiêu lỗi C++ OpenCV SOS
        try:
            img_src = Image.open(img_path).convert('RGB')
        except Exception:
            img_src = str(img_path)

        res1 = model1.predict(source=img_src, imgsz=IMG_SIZE, conf=CONF, max_det=MAX_DET, verbose=False)[0]
        res2 = model2.predict(source=img_src, imgsz=IMG_SIZE, conf=CONF, max_det=MAX_DET, verbose=False)[0]
        img_h, img_w = res1.orig_shape

        reg1 = postprocess_boxes([res1], img_w, img_h, model1.names, IOU_NMS)
        reg2 = postprocess_boxes([res2], img_w, img_h, model2.names, IOU_NMS)
        reg_ens = postprocess_boxes([res1, res2], img_w, img_h, model1.names, IOU_NMS)

        preds_m1[img_path.name] = reg1
        preds_m2[img_path.name] = reg2
        preds_ens[img_path.name] = reg_ens

        m1_writer.writerow([img_path.name, json.dumps(reg1, ensure_ascii=False)])
        m2_writer.writerow([img_path.name, json.dumps(reg2, ensure_ascii=False)])
        ens_writer.writerow([img_path.name, json.dumps(reg_ens, ensure_ascii=False)])

        # Kaggle submission keeps the required schema and omits score.
        clean_reg_ens = [{"bbox": r["bbox"], "type": r["type"], "text": r.get("text", "")} for r in reg_ens]
        sub_writer.writerow([img_path.name, json.dumps(clean_reg_ens, ensure_ascii=False)])

        if idx % 50 == 0:
            print(f"Processed {idx}/{len(image_paths)} images...")

# ---------------------------------------------------------
# COMPUTE mAP WITH OFFICIAL COCOEVAL / PYCOCOTOOLS
# ---------------------------------------------------------
print("\nComputing bbox mAP with official COCOeval...")
res_m1 = evaluate_model_cocoeval(preds_m1, gt_records, "V1.0 (V4.1 only)")
res_m2 = evaluate_model_cocoeval(preds_m2, gt_records, "V1.1 (V4.2 only)")
res_ens = evaluate_model_cocoeval(preds_ens, gt_records, "Ensemble V4.1 + V4.2")

cm_ens = compute_confusion_matrix_hungarian(gt_data, preds_ens, iou_thresh=0.5)

# ---------------------------------------------------------
# WRITE PAPER REPORT
# ---------------------------------------------------------
report_lines = []
report_lines.append("=========================================================================")
report_lines.append("                 DETECTOR RESULTS FOR PAPER (OFFICIAL COCOEVAL)             ")
report_lines.append("=========================================================================")
report_lines.append("")
report_lines.append("Protocol: official pycocotools COCOeval, bbox AP, IoU=0.50:0.95 step 0.05, maxDets=100.")
report_lines.append(f"Candidate generation: conf={CONF}; imgsz={IMG_SIZE}; inference max_det={MAX_DET}; NMS IoU={IOU_NMS}; agnostic_nms=True.")
report_lines.append(f"COCOeval maxDets: {COCO_MAX_DETS}; report category order: {REPORT_CLASSES}.")
report_lines.append(f"YOLO model class-id order: {MODEL_CLASSES}.")
report_lines.append(f"Evaluation images: {res_ens['num_images']}; GT boxes: {res_ens['num_gt_boxes']}; Ensemble pred boxes: {res_ens['num_pred_boxes']}.")
report_lines.append(f"Image set validation: exact GT/inference match = {image_set_summary['exact_match']}.")
report_lines.append(f"GT metadata: {GT_META_PATH}")
report_lines.append(f"Image directory: {TEST_IMG_DIR}")
report_lines.append(f"Model V1 path: {MODEL_PATH_1}")
report_lines.append(f"Model V2 path: {MODEL_PATH_2}")
report_lines.append("Runtime versions: " + ", ".join(f"{k}={v}" for k, v in RUN_INFO.items()))
report_lines.append(
    "Scored prediction artifacts: "
    f"{SCORED_PRED_M1_PATH.name}, {SCORED_PRED_M2_PATH.name}, {SCORED_PRED_ENS_PATH.name}."
)
report_lines.append("")

report_lines.append("### TABLE 1: DETECTOR ABLATION & MODULE RESULTS (OFFICIAL COCOEVAL)")
report_lines.append(f"{'Model Setting':<25} | {'AP@50':<10} | {'AP@50:95':<10} | {'AP@75':<10} | {'AR@100':<10}")
report_lines.append("-" * 76)
for res in [res_m1, res_m2, res_ens]:
    report_lines.append(
        f"{res['model_name']:<25} | {res['map50']:.4f}     | {res['map50_95']:.4f}     | {res['map75']:.4f}     | {res['ar100']:.4f}"
    )
report_lines.append("")

def append_per_class_report_table(report_lines, table_title, result):
    report_lines.append(table_title)
    report_lines.append(f"{'Class Name':<15} | {'AP@50':<10} | {'AP@50:95':<10} | {'GT':<6} | {'Pred':<6}")
    report_lines.append("-" * 63)
    for row in result['per_class']:
        report_lines.append(
            f"{row['class']:<15} | {row['ap50']:.4f}     | {row['ap50_95']:.4f}     | {row['gt']:<6} | {row['pred']:<6}"
        )
    report_lines.append("")

append_per_class_report_table(
    report_lines,
    "### TABLE 2A: PER-CLASS RESULTS - V1.0 (V4.1 ONLY, OFFICIAL COCOEVAL)",
    res_m1,
)
append_per_class_report_table(
    report_lines,
    "### TABLE 2B: PER-CLASS RESULTS - V1.1 (V4.2 ONLY, OFFICIAL COCOEVAL)",
    res_m2,
)
append_per_class_report_table(
    report_lines,
    "### TABLE 2C: PER-CLASS RESULTS - ENSEMBLE V4.1 + V4.2 (OFFICIAL COCOEVAL)",
    res_ens,
)

report_lines.append("\n### TABLE 3: DETECTOR ERROR ANALYSIS (HUNGARIAN MATCHING @ IoU=0.5)")
report_lines.append("- Hungarian matching is only for error analysis, not the primary mAP metric.")
report_lines.append(f"  + Missed annotations: {cm_ens[('annotation', 'missed')]} box.")
report_lines.append(f"  + Formula -> Handwritten confusion: {cm_ens[('formula', 'handwritten')]} box.")
report_lines.append(f"  + Handwritten -> Formula confusion: {cm_ens[('handwritten', 'formula')]} box.")
report_lines.append(f"  + Printed -> Handwritten confusion: {cm_ens[('printed', 'handwritten')]} box.")
report_lines.append(f"  + Handwritten -> Printed confusion: {cm_ens[('handwritten', 'printed')]} box.")
report_lines.append(f"  + Background false positives: {sum(v for k, v in cm_ens.items() if k[0] == 'background')} box.")

report_lines.append("\n### COCOEVAL RAW SUMMARY - ENSEMBLE")
report_lines.append(res_ens['summary'])

report_content = "\n".join(report_lines)
print(report_content)

with open(REPORT_PATH, 'w', encoding='utf-8') as f_rep:
    f_rep.write(report_content)

print(f"\nSaved official COCOeval report to: {REPORT_PATH}")
print(f"Saved scored V1 predictions for metric audit to: {SCORED_PRED_M1_PATH}")
print(f"Saved scored V2 predictions for metric audit to: {SCORED_PRED_M2_PATH}")
print(f"Saved scored ensemble predictions for metric audit to: {SCORED_PRED_ENS_PATH}")
print(f"Saved Kaggle-ready submission to: {SUB_PATH}")

Reading ground-truth metadata...
Found exactly 200 GT images for inference.
Image set validation passed: 200 GT images == 200 inference images.

Loading models...
Model class-id order validation passed: ['handwritten', 'printed', 'formula', 'table', 'annotation', 'image', 'graph']
Report/COCO category order: ['handwritten', 'formula', 'printed', 'annotation', 'table', 'image', 'graph']
Starting inference...
Processed 50/200 images...
Processed 100/200 images...
Processed 150/200 images...
Processed 200/200 images...

Computing bbox mAP with official COCOeval...
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=1.57s).
Accumulating evaluation results...
DONE (t=0.08s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.24s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=1

In [4]:
# import pandas as pd

# df_sub = pd.read_csv(SUB_PATH)
# display(df_sub.head())

# print("\nSố dòng trong submission:", len(df_sub))
# print("Cột:", list(df_sub.columns))